# 1 · The design space

In [1]:
# This is just the code header, ignore it.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dse
import dse.plots as plots
from dse import DATA, WORK

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)


A matrix–vector product on a compute-near-memory system: one rank of **64 DPUs**,
each with up to **24 tasklets** (hardware threads), each DPU owning a slice of
memory (MRAM) and a small working memory (WRAM).


many tasklets, in which order the dimensions are distributed — those are the
*design parameters*. This notebook is about how many there are, and how few
combinations of them are even legal.

In [2]:
mlir = DATA / "small" / "gemv.mlir"
dse.show_mlir(mlir)

The code describes both the platform (`#upmem = #upmem.platform<...>`) and the program (`cinm.op.gemv`).

To run gemv on that platform, we need to 
- Find a suitable DPU and tasklet count
- Tile the loops to create tiles for the MRAM and WRAM memory levels
- Distribute tiles over DPUs/tasklets

These parameters are summarized below:

In [3]:
space = dse.dump_space(mlir) # Ask Cinnamon to create the design space and dump it to json
dse.cinm.params_table(space)

,parameter,kind,values,count,meaning
0,dpus,integer,64 … 64,1,number of DPUs the workgroup spans
1,tasklets,integer,1 … 24,24,tasklets per DPU
2,gemv.M.mram,integer,"1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024",11,mram tile size of iteration dim M
3,gemv.M.wram,integer,1 … 1024,1024,wram tile size of iteration dim M
4,gemv.K.mram,integer,"1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024",11,mram tile size of iteration dim K
5,gemv.K.wram,integer,1 … 1024,1024,wram tile size of iteration dim K
6,gemv.order,permutation,1 … 2,2,which iteration dim occupies which workgroup axis


Let's see how many parameter combinations are in that space:

In [4]:
print(dse.cinm.describe_space(space))

Cartesian product: 6,090,130,000 combinations


That's a lot, but many of them are **infeasible**, in the sense that they couldn't be compiled into a valid program.

For instance, WRAM tiles must be contained in MRAM tiles. That means values of `gemv.M.mram` constrain `gemv.M.wram` (and same for `K`).

Cinnamon can generate these constraints and uses a constraint solver to filter out the infeasible configurations.

In [5]:
print(dse.cinm.describe_space(space, feasible=True))

Cartesian product: 6,090,130,000 combinations
Feasible:          3,234 configurations (5.3e-05 %)
Built in 0.01 s by the constraint solver (6,491 search nodes)


So about 1 in 2 million is feasible!
Below is the list of constraints that were generated and solved. 

In [6]:
pd.set_option("display.max_colwidth", 120)
dse.cinm.constraints_table(space)

,constraint,disposition
0,1024 % gemv.M.mram == 0,static-filter
1,1024 % gemv.K.mram == 0,static-filter
2,(gemv.M.wram | gemv.M.mram),solved
3,(gemv.K.wram | gemv.K.mram),solved
4,(gemv.M.mram | 1024),solved
5,(gemv.K.mram | 1024),solved
6,prod(extent / block) == dpus * tasklets,solved
7,tasklets * sum of operand tiles <= mram (assuming no sharing),solved
8,tasklets * sum of operand tiles <= wram (assuming no sharing),solved
9,"input 0's mram tile must be a whole number of 8-byte DMA granules, i.e. a multiple of 2 elements",solved


## Evaluating configurations


Configurations are the parameters of a valid program. The Cinnamon compiler emits this program, and passes it to a simulator to estimate its latency.

We're looking for the configuration that has the smallest latency among the 3k. Can _you_ find a good one?

The picker below only offers choices that can still complete a **feasible**
configuration — as you fix parameters, the other dropdowns shrink to what remains
legal. Fix all of them, then press **Simulate** to know the latency estimate for that configuration.

Reminder:
- `tasklets`: threads per DPU
- `gemv.M.mram`, `gemv.K.mram`: the tile of `A` each *(DPU, tasklet)* worker
  holds in its MRAM, along the two dimensions of the loop nest
  (M = rows of A / entries of y, K = columns of A / entries of x)
- `gemv.M.wram`, `gemv.K.wram`: the inner tile a tasklet streams through WRAM
- `gemv.order`: which dimension is spread across DPUs first

In [ ]:
import json

# We have computed upfront the latency for every configuration and put it in this json file.
feasible = dse.load_pool(DATA / "small" / "pool.csv")[lambda p: dse.param_columns(p)]


def simulate(config):
    report = dse.eval_solution(mlir, config)
    # Notebook 2 reads this back to show where the guess landed.
    (WORK / "my_guess.json").write_text(
        json.dumps({"config": config, "cost_ms": report.total_ms}, indent=1)
    )
    return report


picker = dse.ConfigPicker(feasible, space, on_pick=simulate).show()

The breakdown says where the time goes: the kernel itself, launching it,
moving the operands in and the result back. Keep the total in mind.

Skipped the picker? The cell below falls back to a plausible first guess —
all 16 tasklets, middling tiles — so the rest of the tutorial has something to
compare against.

In [8]:
if picker.result is None:
    report = simulate(picker.default_guess())
    print("no configuration simulated in the widget; used a default guess:", report.config)
else:
    report = picker.result
report

no configuration simulated in the widget; used a default guess: {'dpus': 64, 'tasklets': 16, 'gemv.M.mram': 32, 'gemv.M.wram': 4, 'gemv.K.mram': 32, 'gemv.K.wram': 32, 'gemv.order[0]': 2, 'gemv.order[1]': 1}


kernel,0.777
kernel.launchOverhead,0.046
cpu.compact,6.658
cpu.other,0.013
transfer.array,0.339
transfer_back.array,0.103


The cost estimation is a breakdown:
- kernel: the output of the cycle-accurate simulator
- kernel.launchOverhead: an additional overhead term, calibrated on the test machine
- cpu.compact: data rearrangement before transfer
- cpu.other: all other host CPU time, including reduction loops on the host
- transfer.array: transferring data onto the DPU array
- transfer_back.array: transferring the result back


In the next notebook you will see where that number lands among *all*
feasible configurations.


**What to take away.** The design space falls out of the program's structure and the hardware's structure, and is pruned using feasibility constraints. The same derivation, with a different platform description, gives the space of a
different CNM system.

→ Notebook 2: what the configurations look like.